# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/malaikaansar/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 24), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.84 MiB | 8.48 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [3]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking and Classification:**
Our primary question is "Which page should an editor fix first?". Therefore, this is fundamentally a Ranking task aimed at building a priority queue. To power this ranking, we will use a Classification model to predict the probability of a page declining.

In [ ]:
# No code required for this section. The ML task type is defined in the markdown above.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy Target:** is_declining_label (***where trend_direction == 'down')***.
An ideal ML target should be an "observed future outcome." However, in this dataset snapshot, we are using the current downward trend as a proxy target. Because this target is derived from a defined rule ***(trend_pct***), we must strictly exclude both trend_direction and trend_pct from our features to prevent data leakage.

In [ ]:
# The target definition will be executed in the Unit of Analysis step below.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K (specifically, Precision@50):**
Since our goal is to build a prioritized queue, we care more about the accuracy of our top recommendations than the overall accuracy of the model. If the review team has the capacity to update 50 pages a week, Precision@50 will accurately measure how many of our top 50 ranked pages were genuinely declining and needed intervention.

In [ ]:
# The success metric is defined in the markdown above.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row** = One Pseudonymized Content Item (Webpage)
In the code below, we load the anonymized dataset, create our proxy target, and drop the leakage-prone columns to clearly isolate our unit of analysis.

In [7]:
import pandas as pd
import os

# Define the relative path to the dataset within the cloned repository
# (Assuming the notebook is running from the root of the cloned repo)
file_path = 'data/raw/content_refresh_anonymized.csv'

try:
    # Ingest the raw anonymized dataset
    df = pd.read_csv(file_path)

    # Formulate the proxy target variable for the classification task
    # 1 indicates a downward performance trend (risk of decline), 0 otherwise
    df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)

    # Isolate the core columns that establish our unit of analysis
    # CRITICAL: 'trend_direction' and 'trend_pct' are strictly omitted here to prevent target leakage
    feature_subset = [
        'content_id',
        'client_id',
        'content_type',
        'impressions_90d',
        'ctr',
        'content_age_days',
        'target_is_declining'
    ]

    unit_df = df[feature_subset]

    # Output the structural grain of the dataset to validate the framing
    print("Unit of Analysis: 1 Row = 1 Pseudonymized Content Item (Webpage)")
    display(unit_df.head())

except FileNotFoundError:
    # Graceful error handling for environment pathing issues
    print(f"Error: Dataset not found at the specified path: '{file_path}'")
    print(f"Current Working Directory: {os.getcwd()}")
    print("Please ensure the repository is cloned correctly and the path matches your environment structure.")

Unit of Analysis: 1 Row = 1 Pseudonymized Content Item (Webpage)


,content_id,client_id,content_type,impressions_90d,ctr,content_age_days,target_is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,3803,0.76,187,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,0.05,445,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,0.09,141,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,0.49,463,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,0.13,263,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A plain rule (like an if-statement) fails here because the search signals impacting page performance (age, position, CTR, volume) are tangled and shift over time. Furthermore, missing data patterns (such as specific content types consistently missing keyword data) make rules even more brittle. ML can learn directly from these messy, complex signals to output a nuanced priority score that hard-coded thresholds would completely miss.

In [ ]:
# The reasoning is explained in the markdown above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.